In [23]:
import csv
import os
import pickle
import time

from collections import defaultdict
from dataclasses import dataclass
from itertools import groupby
from operator import itemgetter
from typing import List

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pytorch_lightning as pl
import torch
import torch.nn as nn
import torch.nn.functional as F
import yaml

from torch.utils.data import DataLoader, Dataset
from torch_geometric.data import Data
from torch_geometric.nn.conv import MessagePassing

from tqdm import tqdm

import utils
from training.logger import Logger

In [24]:
model_config = {
    'embedding_dim': 32,
    'dropout': 0.5,
}

train_config = {
    'dataset_name': 'YAGO',  # Change to: 'GDELT', 'ICEWS14', 'ICEWS18', 'WIKI'
    'batch_size': 1,
    'max_epochs': 30,
    'valid_epochs': 5,
    'lr': 0.001,
    'weight_decay': 1e-5,
    'gpu_id': '0',
    'save_dir': 'SAVE',
}

dataset_config = {
    "dataset": "YAGO",
    "history_len": 5,
    "dilate_len": 1,
    "add_inverse": True
}

# Dataset

In [25]:
@dataclass
class Quadruple:
    src: int
    rel: int
    dst: int
    tim: int

    def inverse(self, num_rels: int):
        return Quadruple(src=self.dst, rel=self.rel + num_rels, dst=self.src, tim=self.tim)

    def to_tensor(self):
        return torch.tensor([self.src, self.rel, self.dst, self.tim], dtype=torch.long)

    @classmethod
    def from_tensor(cls, tensor: torch.Tensor):
        return cls(src=int(tensor[0]), rel=int(tensor[1]), dst=int(tensor[2]), tim=int(tensor[3]))

    def as_tuple(self):
        return (self.src, self.rel, self.dst, self.tim)

In [26]:
class ReadableTKG:
    def __init__(self, entity2id_path=None, relation2id_path=None):
        self.entity2id = self.load_vocab(entity2id_path)
        self.relation2id = self.load_vocab(relation2id_path)
        self.id2entity = {v: k for k, v in self.entity2id.items()}
        self.id2relation = {v: k for k, v in self.relation2id.items()}

    @staticmethod
    def load_vocab(path):
        if path is None or not os.path.exists(path):
            return {}
        vocab = {}
        with open(path, "r", encoding="utf-8") as f:
            for line_idx, line in enumerate(f):
                parts = line.rstrip("\n").split("\t")
                if len(parts) < 2:
                    continue
                try:
                    vocab[parts[0]] = int(parts[1])
                except ValueError:
                    print(f"[Warning] Skip malformed line {line_idx}: {line.strip()}")
        return vocab

    def add_inverse_relations(self, num_rels: int):
        for rel, idx in list(self.relation2id.items()):
            self.id2relation[idx + num_rels] = f"Inverse_{rel}"

    def quadruple_to_string(self, quadruple):
        if isinstance(quadruple, torch.Tensor):
            quadruple = Quadruple.from_tensor(quadruple)

        src = self.id2entity.get(quadruple.src, str(quadruple.src))
        rel = self.id2relation.get(quadruple.rel, str(quadruple.rel))
        dst = self.id2entity.get(quadruple.dst, str(quadruple.dst))

        return f"({src}) --[{rel}]--> ({dst}) @ t={quadruple.tim}"

    def snapshot_to_string(self, snapshot, max_lines=None):
        lines = [self.quadruple_to_string(row) for row in snapshot]
        if max_lines is not None:
            lines = lines[:max_lines]

        return "\n".join(lines)

In [27]:
class TemporalKnowledgeGraph:
    def __init__(self, quadruples: List, num_nodes: int, num_rels: int, add_inverse: bool = True):
        self.num_nodes = num_nodes
        self.raw_num_rels = num_rels
        self.add_inverse = add_inverse

        if add_inverse:
            quadruples = quadruples + [q.inverse(num_rels) for q in quadruples]
            self.num_rels = num_rels * 2
        else:
            self.num_rels = num_rels

        self.quadruples = sorted(quadruples, key=lambda x: x.tim)
        self.snapshots = self.build_snapshots()

    def build_snapshots(self):
        snapshots = []

        for _, group in groupby(self.quadruples, key=lambda x: x.tim):
            group = list(group)

            src = torch.tensor([q.src for q in group], dtype=torch.long)
            rel = torch.tensor([q.rel for q in group], dtype=torch.long)
            dst = torch.tensor([q.dst for q in group], dtype=torch.long)

            edge_index = torch.stack([src, dst], dim=0)
            snapshot = Data(edge_index=edge_index, edge_attr=rel, num_nodes=self.num_nodes)
            snapshots.append(snapshot)

        return snapshots

    @property
    def num_snapshots(self):
        return len(self.snapshots)

    def get_snapshot(self, idx: int):
        return self.snapshots[idx]

    def get_history(self, idx: int, history_len: int, dilate_len: int):
        history_indices = [idx - i * dilate_len for i in range(history_len, 0, -1)]

        return [self.snapshots[i] for i in history_indices]

    def stat(self, name="Graph"):
        num_edges = [snapshot.edge_index.size(1) for snapshot in self.snapshots]

        return (
            f"[{name}] "
            f"Nodes: {self.num_nodes}, "
            f"Relations: {self.num_rels}, "
            f"Snapshots: {self.num_snapshots}, "
            f"Max edges: {max(num_edges)}, "
            f"Min edges: {min(num_edges)}"
        )

    def __str__(self):
        return self.stat()

    def __repr__(self):
        return self.stat()

In [28]:
import os
from itertools import groupby
from typing import List

import torch
from torch.utils.data import Dataset, DataLoader
from torch_geometric.data import Data

class TKGDataset(Dataset):
    def __init__(self, dataset="YAGO", mode="train", history_len=5, dilate_len=1, add_inverse=True):
        super().__init__()
        self.dataset_name = dataset
        self.mode = mode
        self.history_len = history_len
        self.dilate_len = dilate_len

        self.data_path = os.path.join("./data", dataset)
        self.num_nodes, self.raw_num_rels, self.num_times = utils.get_total_number(self.data_path, "stat.txt")
        data, _ = utils.load_quadruples(self.data_path, f"{mode}.txt")

        quadruples = [
            Quadruple(int(r[0]), int(r[1]), int(r[2]), int(r[3]))
            for r in data
        ]

        self.graph = TemporalKnowledgeGraph(quadruples=quadruples, num_nodes=self.num_nodes, num_rels=self.raw_num_rels, add_inverse=add_inverse)
        self.num_rels = self.graph.num_rels
        self.times = list(range(self.graph.num_snapshots))

    def __len__(self):
        return len(self.times) - self.history_len * self.dilate_len

    def __getitem__(self, idx):
        idx += self.history_len * self.dilate_len
        history = self.graph.get_history(idx=idx, history_len=self.history_len, dilate_len=self.dilate_len)
        target = self.graph.get_snapshot(idx)
        return history, target

    def collate_fn(self, batch):
        histories, targets = zip(*batch)
    
        return list(histories), list(targets)

    def get_loader(self, batch_size=1, num_workers=0):
        return DataLoader(
            self,
            batch_size=batch_size,
            shuffle=(self.mode == "train"),
            num_workers=num_workers,
            collate_fn=self.collate_fn
        )

    def stat(self):
        return self.graph.stat(f"{self.dataset_name}-{self.mode}")

    def __str__(self):
        return self.stat()

    def __repr__(self):
        return self.stat()

In [29]:
class TKGLightningModule(pl.LightningModule):
    def __init__(self, model, learning_rate: float, weight_decay: float, save_dir='SAVE', dataset_name='YAGO', use_cuda=True):
        super().__init__()
        self.model = model
        self.dataset_name = dataset_name
        self.lr = learning_rate
        self.weight_decay = weight_decay
        self.device_str = f'cuda' if (torch.cuda.is_available() and use_cuda) else 'cpu'

        # Keep a dedicated application logger (your Logger)
        self.app_logger = Logger(save_dir, dataset_name)
        self.main_dir = self.app_logger.get_log_dir()
        self.model_path = os.path.join(self.main_dir, 'models')
        os.makedirs(self.model_path, exist_ok=True)

        # Best metric tracking
        self.best_mrr = 0.0
        
        # CSV metrics logging
        self.metrics_file = os.path.join(self.main_dir, 'metrics.csv')
        self._metrics_csv_f = None
        self._metrics_writer = None
        self._init_metrics_csv()

    def _init_metrics_csv(self):
        self._metrics_csv_f = open(self.metrics_file, 'w', newline='')
        self._metrics_writer = csv.writer(self._metrics_csv_f)
        self._metrics_writer.writerow(['Epoch', 'Train Loss', 'Val Loss'])
        self._metrics_csv_f.flush()

    def log_config(self, model_config: dict = None, training_config: dict = None, dataset_config: dict = None):
        self.app_logger.write("Model Configuration:")
        if model_config:
            for key, value in sorted(model_config.items()):
                self.app_logger.write(f"  {key:20s} = {value}")
        self.app_logger.write("Training Configuration:")
        if training_config:
            for key, value in sorted(training_config.items()):
                self.app_logger.write(f"  {key:20s} = {value}")
        self.app_logger.write("Dataset Configuration:")
        if dataset_config:
            for key, value in sorted(dataset_config.items()):
                self.app_logger.write(f"  {key:20s} = {value}")
        self.app_logger.write("")

    def forward(self, history_graphs):
        return self.model(history_graphs)

    def training_step(self, batch, batch_idx):
        history_graphs = batch.get('history_graphs', None) if isinstance(batch, dict) else None
        if history_graphs is None:
            return None
        
        logits = self.forward(history_graphs)
        if logits is None:
            return None
        
        # Create labels (all positive samples)
        labels = torch.ones_like(logits, dtype=logits.dtype, device=logits.device)
        loss = F.binary_cross_entropy_with_logits(logits, labels)
        self.log('train/loss', loss, on_step=False, on_epoch=True, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        history_graphs = batch.get('history_graphs', None) if isinstance(batch, dict) else None
        if history_graphs is None:
            return None
        
        logits = self.forward(history_graphs)
        if logits is None:
            return None
        
        labels = torch.ones_like(logits, dtype=logits.dtype, device=logits.device)
        loss = F.binary_cross_entropy_with_logits(logits, labels)
        self.log('val/loss', loss, on_step=False, on_epoch=True, prog_bar=True)
        return loss

    def on_train_epoch_end(self):
        train_loss = self.trainer.callback_metrics.get('train/loss')
        if train_loss is not None:
            train_loss = float(train_loss)
            self.app_logger.write(f"[TRAIN] Epoch {self.current_epoch + 1}: Loss = {train_loss:.6f}")
        
        if torch.cuda.is_available():
            mem_info = utils.get_gpu_memory_info(self.device_str)
            if mem_info:
                self.app_logger.write(
                    f"[VRAM] Peak: {mem_info['peak']:.2f}GB | Reserved: {mem_info['reserved']:.2f}GB"
                )

    def on_validation_epoch_end(self):
        train_loss = self.trainer.callback_metrics.get('train/loss')
        val_loss = self.trainer.callback_metrics.get('val/loss')
        train_loss = float(train_loss) if train_loss is not None else None
        val_loss = float(val_loss) if val_loss is not None else None
        
        # Write to CSV
        self._metrics_writer.writerow([
            self.current_epoch + 1,
            f'{train_loss:.6f}' if train_loss is not None else '',
            f'{val_loss:.6f}' if val_loss is not None else ''
        ])
        self._metrics_csv_f.flush()

    def configure_optimizers(self):
        opt = torch.optim.Adam(self.model.parameters(), lr=self.lr, weight_decay=self.weight_decay)
        return opt

    def on_fit_end(self):
        if self._metrics_csv_f:
            self._metrics_csv_f.close()
        self.app_logger.write("Training completed!")

# Model

In [30]:
class EmbeddingLayer(nn.Module):
    def __init__(self, num_nodes: int, num_rels: int, embedding_dim: int, L: int):
        super().__init__()

        self.s_embedding = nn.Embedding(num_nodes, embedding_dim)
        self.o_embedding = nn.Embedding(num_nodes, embedding_dim)
        self.rel_embedding = nn.Embedding(num_rels, embedding_dim)
        self.time_embedding = nn.Embedding(L, embedding_dim)

    def forward(self, s: torch.Tensor, o: torch.Tensor, r: torch.Tensor):
        s_embed = self.s_embedding(s)
        o_embed = self.o_embedding(o)
        rel_embed = self.rel_embedding(r)

        return s_embed, o_embed, rel_embed

In [31]:
class RGCNConv(MessagePassing):
    def __init__(self, in_channels, out_channels, num_relations, num_bases):
        super().__init__(aggr="mean")
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.num_relations = num_relations
        self.num_bases = num_bases

        self.basis = nn.Parameter(torch.empty(num_bases, in_channels, out_channels))
        self.att = nn.Parameter(torch.empty(num_relations, num_bases))
        self.root = nn.Parameter(torch.empty(in_channels, out_channels))
        self.bias = nn.Parameter(torch.empty(out_channels))
        self.reset_parameters()

    def reset_parameters(self):
        nn.init.xavier_uniform_(self.basis)
        nn.init.xavier_uniform_(self.att)
        nn.init.xavier_uniform_(self.root)
        nn.init.zeros_(self.bias)

    def forward(self, x, edge_index, edge_type, edge_norm=None, size=None):
        weight = torch.einsum("rb,bio->rio", self.att, self.basis)
        return self.propagate(edge_index, size=size, x=x, weight=weight, edge_type=edge_type, edge_norm=edge_norm)

    def message(self, x_j, edge_type, weight, edge_norm):
        if x_j is None:
            raise NotImplementedError("Featureless mode is not supported.")

        out = torch.einsum("ei,eio->eo", x_j, weight[edge_type])
        if edge_norm is not None:
            out = out * edge_norm.unsqueeze(-1)
        return out

    def update(self, aggr_out, x):
        if x is not None:
            out = aggr_out + x @ self.root
        else:
            out = aggr_out + self.root.sum(dim=0)
        out = out + self.bias
        return out

    def __repr__(self):
        return f"{self.__class__.__name__}({self.in_channels}, {self.out_channels}, num_relations={self.num_relations})"

In [32]:
class Dismult(nn.Module):
	def __init__(self, num_nodes, num_rels, embedding_dim):
		super().__init__()

	def forward(self, entity_src: torch.Tensor, rel_embed: torch.Tensor, entity_dst: torch.Tensor):
		score = torch.sum(entity_src * rel_embed * entity_dst, dim=-1)
		return score

In [33]:
class Model(nn.Module):
    def __init__(self, num_nodes: int, num_rels: int, embedding_dim: int, L: int, device: str = "cuda"):
        super().__init__()
        self.num_nodes = num_nodes
        self.num_rels = num_rels
        self.embedding_dim = embedding_dim
        self.embedding_layer = EmbeddingLayer(num_nodes, num_rels, embedding_dim, L=L)
        self.dismult = Dismult(num_nodes, num_rels, embedding_dim)
        self.device = device

    def get_params(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

    def forward(self, history_graphs):
        print(history_graphs)
        edge_index = [h.edge_index for h in history_graphs[0]]
        print(len(edge_index))
        
        s_embedding, o_embedding, rel_embedding = self.embedding_layer(s, o, r)
        score = self.dismult(s_embedding, rel_embedding, o_embedding)
    
        return score

In [34]:
train_dataset = TKGDataset(**dataset_config, mode="train")
valid_dataset = TKGDataset(**dataset_config, mode="valid")
test_dataset = TKGDataset(**dataset_config, mode="test")

In [35]:
model = Model(
    train_dataset.num_nodes,
    train_dataset.num_rels,
    embedding_dim=model_config['embedding_dim'],
    L=5,
)
model = model.to(device)

num_trainable_params = model.get_params()
print(f"Number of trainable parameters: {num_trainable_params:,}")

Number of trainable parameters: 680,672


In [40]:
# for history_graphs, target_graph in train_dataset.get_loader(batch_size=1):
#     print(type(history_graphs), len(history_graphs), len(history_graphs[0]))
#     # print(type(target_graph), len(target_graph), target_graph[0])
#     print(len(history_graphs), len(target_graph))
#     break

# # print(

<class 'list'> 1 5
1 1


In [ ]:
# Setup Lightning Module and Trainer
tkg_module = TKGLightningModule(
    model=model,
    learning_rate=train_config['lr'],
    weight_decay=train_config['weight_decay'],
    save_dir=train_config['save_dir'],
    dataset_name=train_config['dataset_name'],
)

# Log configuration
tkg_module.log_config(model_config, train_config, dataset_config)

# Create data loaders - external dataloaders passed to trainer
train_loader = train_dataset.get_loader(batch_size=train_config['batch_size'], num_workers=0)
valid_loader = valid_dataset.get_loader(batch_size=train_config['batch_size'], num_workers=0)

print(f"Train loader size: {len(train_loader)}")
print(f"Valid loader size: {len(valid_loader)}")

In [ ]:
# Create PyTorch Lightning Trainer
trainer = pl.Trainer(
    max_epochs=train_config['max_epochs'],
    accelerator='gpu' if torch.cuda.is_available() else 'cpu',
    devices=1,
    logger=False,  # We use custom logging via Logger
    enable_checkpointing=False,  # Manual checkpoint saving in module
    enable_model_summary=False,
    enable_progress_bar=True,
    # log_every_n_steps=10,
)

# Fit trainer with external dataloaders
trainer.fit(
    model=tkg_module,
    train_dataloaders=train_loader,
    # val_dataloaders=valid_loader
)

print("Training completed!")
print(f"Model path: {tkg_module.model_path}")

In [ ]:
config_dst_dir = os.path.join(main_dirName, 'configs')
os.makedirs(config_dst_dir, exist_ok=True)

# Combine all configs into one dictionary
combined_config = {
    'model': model_config,
    'training': train_config,
    'dataset': dataset_config,
}

# Save combined config to single YAML file
config_file = os.path.join(config_dst_dir, 'config.yaml')
with open(config_file, 'w') as f:
    yaml.dump(combined_config, f, default_flow_style=False)
print(f"Combined config saved to: {config_file}")

In [ ]:
metrics_file = os.path.join(main_dirName, 'metrics.csv')

if os.path.exists(metrics_file):
    # Read metrics from CSV
    df = pd.read_csv(metrics_file)
    
    # Create figure
    fig, ax = plt.subplots(figsize=(10, 6))
    
    # Plot training loss
    ax.plot(df['Epoch'], df['Train Loss'], 'b-o', label='Train Loss', linewidth=2, markersize=4)
    ax.set_xlabel('Epoch', fontsize=12)
    ax.set_ylabel('Loss', fontsize=12)
    ax.set_title(f'Training Loss - {dataset_config["dataset"]}', fontsize=14)
    
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)
    
    # Save plot
    plot_file = os.path.join(main_dirName, 'loss_plot.png')
    plt.savefig(plot_file, dpi=150, bbox_inches='tight')
    print(f"Loss plot saved to: {plot_file}")
    
    # Display plot
    plt.show()
else:
    print(f"Metrics file not found: {metrics_file}")

In [ ]:
import matplotlib.pyplot as plt

metrics_file = os.path.join(main_dirName, 'metrics.csv')

if os.path.exists(metrics_file):
    # Read metrics from CSV
    df = pd.read_csv(metrics_file)
    
    # Create figure
    fig, ax = plt.subplots(figsize=(10, 6))
    
    # Filter out NaN values for Val Loss
    valid_data = df.dropna(subset=['Val Loss'])
    
    # Plot validation loss (only non-NaN points)
    ax.plot(valid_data['Epoch'], valid_data['Val Loss'], 'r-o', label='Val Loss', linewidth=2, markersize=6)
    ax.set_xlabel('Epoch', fontsize=12)
    ax.set_ylabel('Loss', fontsize=12)
    ax.set_title(f'Validation Loss - {dataset_config["dataset"]}', fontsize=14)
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)
    
    # Save plot
    plot_file = os.path.join(main_dirName, 'val_loss_plot.png')

    plt.savefig(plot_file, dpi=150, bbox_inches='tight')
    print(f"Loss plot saved to: {plot_file}")
    
    # Display plot
    plt.show()
else:
    print(f"Metrics file not found: {metrics_file}")